# 列生成(基礎・双対安定化・price-and-branch)— 適用前・原理・適用・効果

「取りうるパターン」(裁断パターン、シフトパターン、経路など)が指数的に多いモデルは、
全パターンを列挙した**コンパクト定式化**をそのまま書き下せる/解けるとは限らない。
**列生成**は、制限主問題(RMP: 現在持っている列だけの LP)と**pricing問題**(改善列を
1本探す小さな最適化)を交互に解き、必要な列だけをその都度作ることでこれを避ける。

この notebook は次の型で列生成を追う。

1. **課題(before)** — 全パターン列挙のコンパクト定式化(Kantorovich)の弱さ
2. **原理(principle)** — RMP↔pricingの反復で LP境界がどう単調に締まっていくか(実測)
3. **適用(how)** — `mk.column_generation` / `mk.price_and_branch` を最小構成で確認する
4. **効果(before/after)** — コンパクト定式化 vs 列生成後の制限主問題(整数)を比較する

題材は **カッティングストック**(`samples/packing_and_cutting/cutting_stock.py`)。
ロール幅100から品目(幅, 需要)を切り出す、使用ロール数を最小化する古典的な列生成の題材。

In [1]:
import sys, pathlib
# リポジトリルート(pyproject.toml を持つ階層)を探して import パスに載せる。
# docs/samples/ が存在するため "samples" 有無では docs で止まる。pyproject.toml を目印にする。
ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").is_file() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
for sub in ["samples", "samples/packing_and_cutting"]:
    p = str(ROOT / sub)
    if p not in sys.path:
        sys.path.insert(0, p)

import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import HTML, display
from pyscipopt import Model, quicksum

def show(fig):  # 静的サイトに埋め込める形でグラフを表示(CDN の plotly.js を読む)
    display(HTML(fig.to_html(include_plotlyjs="cdn", full_html=False,
                             config={"displayModeBar": False})))

import minlpkit as mk
import cutting_stock as cs

# dataviz 参照パレット(minlpkit.live.plots と統一)
C = dict(ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9",
         axis="#c3c2b7", surface="#fcfcfb", s1="#2a78d6", s2="#008300", warn="#c25a00")
FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'

def base_layout(title, xtitle, ytitle, height=380):
    ax = dict(gridcolor=C["grid"], linecolor=C["axis"],
              tickfont=dict(color=C["muted"], size=11),
              title_font=dict(color=C["ink2"], size=12), zeroline=False)
    return go.Layout(
        title=dict(text=title, font=dict(color=C["ink"], size=15, family=FONT), x=0.01),
        paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
        font=dict(family=FONT, color=C["ink2"], size=12),
        xaxis=dict(ax, title=xtitle), yaxis=dict(ax, title=ytitle),
        margin=dict(l=60, r=20, t=48, b=48), height=height, hovermode="closest",
        legend=dict(orientation="h", yanchor="bottom", y=1.0, x=1.0, xanchor="right",
                    font=dict(size=11, color=C["ink2"]), bgcolor="rgba(0,0,0,0)"))
print("repo root:", ROOT)

repo root: C:\work_space\mip


## 1. 課題(before) — コンパクト定式化(Kantorovich)を素朴に解く

`cs.build_compact()` はロール `r` ごとに使う/使わないの `y_r` とパターン `a_{i,r}` を持つ
コンパクト定式化。パターンという概念を持たず「どの品目をどのロールに何本詰めるか」を
直接書き下すため、**ロールの入れ替えに対して対称**(どのロール番号を使っても等価)になる。
`mk.analyze` で診断すると、対称性(参考情報・SCIPが自動処理)は検出されるものの、
その対称性を処理してなお **5000超のノード** を要することが分かる。

In [2]:
report = mk.analyze(cs.build_compact, name="cutting-stock-compact", time_limit=20)
print(report.summary())
print("探索ノード数:", report.metrics.get("nodes"), " 最終gap:", report.metrics.get("gap"),
      " 対称群の数:", report.metrics.get("n_sym_groups"),
      " 最大対称群サイズ:", report.metrics.get("largest_sym_group"))

[cutting-stock-compact] 検出症状 1件:
  [good] 入替可能な変数群(対称性)を検出(参考情報) -> 通常は対応不要(SCIPが自動処理)。usesymmetryを無効化した運用でのみ辞書式除去が有効
探索ノード数: 5231  最終gap: 0.0  対称群の数: 7  最大対称群サイズ: 60


最大60本のロールが互いに入れ替え可能な対称群としてモデルに入っている。SCIPは対称性を
自動検出して分枝を減らす(`good` finding)が、それでも解くのに数千ノードを要する。
**パターンを列として扱い、必要な列だけ生成する**列生成が実務でこの種のモデルに使われる
理由がここにある。

## 2. 原理(principle) — RMP↔pricingの反復で LP境界が締まる過程(実測)

制限主問題(RMP)$\min \sum_p \lambda_p \ \text{s.t.}\ \sum_p \mathrm{col}_p \lambda_p \ge \mathrm{demand},\ \lambda\ge0$ を解いて双対 $\pi$ を得る。
$\pi$ のもとで最も価値のある切り出しパターンを **pricingナップサック**
$\max \sum_i \pi_i a_i \ \text{s.t.}\ \sum_i w_i a_i \le W$ で1本探し、被約コスト(`pricing値 > 1`)が負なら
RMP に追加する — を繰り返す。pricing値が1以下になれば改善パターンなし=LP最適。

```
  RMP min Σλ s.t. Σcol·λ>=demand ──双対π──▶ pricing max Σπ_i·a_i s.t. Σw_i·a_i<=W
       ▲                                                    │
       └──────── 被約コスト<0(pricing値>1)の新しい列を追加 ─────┘
  pricing値<=1(改善列なし)で終了 = LP最適
```

初期列は「各品目だけを詰めた自明パターン」から始める。

In [3]:
def pricing(duals):
    kp = Model(); kp.hideOutput()
    a = [kp.addVar(vtype="I", lb=0, name=f"a{i}") for i in range(cs.N_ITEMS)]
    kp.addCons(quicksum(cs.WIDTHS[i] * a[i] for i in range(cs.N_ITEMS)) <= cs.W)
    kp.setObjective(quicksum(duals[i] * a[i] for i in range(cs.N_ITEMS)), "maximize")
    kp.optimize()
    return [round(kp.getVal(v)) for v in a], kp.getObjVal()


init_cols = [[cs.W // cs.WIDTHS[i] if j == i else 0 for j in range(cs.N_ITEMS)]
             for i in range(cs.N_ITEMS)]
rhs = [float(d) for d in cs.DEMANDS]

res = mk.column_generation(rhs, init_cols, pricing, alpha=0.0)
print(f"GG LP境界={res['lp_bound']:.3f}  反復={len(res['history'])}  生成列数={res['n_cols']}")

GG LP境界=23.545  反復=8  生成列数=13


In [4]:
def count_patterns(widths, W):
    ws = sorted(set(widths))
    def rec(idx, rem):
        if idx == len(ws):
            return 1
        return sum(rec(idx + 1, rem - k * ws[idx]) for k in range(rem // ws[idx] + 1))
    return rec(0, W) - 1

total_patterns = count_patterns(cs.WIDTHS, cs.W)
material_lb = sum(cs.WIDTHS[i] * cs.DEMANDS[i] for i in range(cs.N_ITEMS)) / cs.W

its = [h["iter"] for h in res["history"]]
fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.12,
    subplot_titles=("RMPのLP境界の収束(列を追加するたび締まる)",
                    "pricing値(1.0以下で改善列なし=収束)"))
fig.add_trace(go.Scatter(x=its, y=[h["lp_bound"] for h in res["history"]], mode="lines+markers",
    line=dict(color=C["s1"], width=2), marker=dict(size=7, color=C["s1"]),
    name="LP境界", hovertemplate="反復%{x}<br>LP境界 %{y:.3f}<extra></extra>"), row=1, col=1)
fig.add_hline(y=material_lb, line=dict(color=C["muted"], width=2, dash="dash"),
              annotation_text=f"材料下界 {material_lb:.2f}", row=1, col=1,
              annotation_font=dict(color=C["ink2"], size=11))
fig.add_trace(go.Scatter(x=its, y=[h["pricing_val"] for h in res["history"]], mode="lines+markers",
    line=dict(color=C["s2"], width=2), marker=dict(size=7, color=C["s2"]),
    name="pricing値", hovertemplate="反復%{x}<br>pricing値 %{y:.3f}<extra></extra>"), row=1, col=2)
fig.add_hline(y=1.0, line=dict(color=C["warn"], width=2, dash="dash"),
              annotation_text="収束閾値1.0", row=1, col=2,
              annotation_font=dict(color=C["ink2"], size=11))
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=380, showlegend=False,
    margin=dict(l=50, r=20, t=56, b=44))
fig.update_xaxes(title_text="列生成の反復", gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
show(fig)

LP境界は単調に締まりながら材料下界(理論上の最良LP下界)に到達し、pricing値が1.0まで
下がったところで「改善パターンなし」と判定して止まる。**総実行可能パターン数**に対して
生成した列数の割合を見ると、列生成の本質(「必要な列だけをその都度作る」)が数字で見える。

In [5]:
gen = res["n_cols"]
print(f"総実行可能パターン数: {total_patterns}")
print(f"列生成が生成した列数: {gen} ({gen/total_patterns*100:.1f}%)")
print(f"GG LP境界={res['lp_bound']:.3f}  材料下界={material_lb:.3f}  "
      f"(ほぼ一致 = 全列挙しなくても最良のLP下界に到達)")

総実行可能パターン数: 131
列生成が生成した列数: 13 (9.9%)
GG LP境界=23.545  材料下界=23.520  (ほぼ一致 = 全列挙しなくても最良のLP下界に到達)


## 3. 適用(how) — `mk.column_generation` / `mk.price_and_branch`

問題固有の部分は `pricing_fn(duals) -> (column, value)` だけ。ここでは API の契約を
最小構成で確認する(2節と同じ pricing だが、収束後の LP境界が真の整数最適の下界として
妥当であることだけを見る)。

In [6]:
res_min = mk.column_generation(rhs, init_cols, pricing, alpha=0.0)
print(f"lp_bound={res_min['lp_bound']:.3f} は整数最適の下界(<= 真の整数最適)であるべき: "
      f"{res_min['lp_bound'] <= 24.0 + 1e-6}")

bnp = mk.price_and_branch(rhs, init_cols, pricing, alpha=0.0)
optimal_proved = abs(bnp["lp_lb"] - bnp["int_obj"]) < 1e-6
print(f"price_and_branch: lp_lb={bnp['lp_lb']}  int_obj={bnp['int_obj']:.3f}  "
      f"lp_lb==int_obj(最適性証明)={optimal_proved}")

lp_bound=23.545 は整数最適の下界(<= 真の整数最適)であるべき: True
price_and_branch: lp_lb=24  int_obj=24.000  lp_lb==int_obj(最適性証明)=True


`price_and_branch` は生成済みの列**だけ**を使った整数制限主問題を解く。これは
**上界のみ**保証する(真の整数最適 $\le$ この解、とは限らない)。`lp_lb == int_obj` が
成り立って初めて最適性が証明される — この題材ではロール24本で成立している。

## 4. 効果(before/after) — コンパクト定式化 vs 列生成後の制限主問題

`mk.compare_variants` に、(a) 1節のコンパクト定式化(全パターンを陽に持つ)と
(b) 列生成が生成した列**だけ**を使った制限主問題(整数)を渡し、
ルート双対境界・最終gap・ノード数を比較する。(b) は2節で生成した一部のパターンしか
持たないのに、コンパクト定式化と同じ整数最適に到達するかどうかを見る。

In [7]:
def build_rmp_ip():
    cols = res["columns"]
    m = Model("cutting_stock_rmp_ip")
    lam = [m.addVar(vtype="I", lb=0, name=f"lam_{p}") for p in range(len(cols))]
    for i in range(cs.N_ITEMS):
        m.addCons(quicksum(cols[p][i] * lam[p] for p in range(len(cols))) >= rhs[i],
                  name=f"demand_{i}")
    m.setObjective(quicksum(lam), "minimize")
    m.data = dict(lam=lam, cols=cols)
    return m

df = mk.compare_variants(
    {"compact(全パターン陽に列挙)": cs.build_compact,
     f"RMP-IP(列生成の{gen}列だけ)": build_rmp_ip},
    time_limit=20)
df[["variant", "root_dual", "final_dual", "final_gap", "nodes"]]

,variant,root_dual,final_dual,final_gap,nodes
0,compact(全パターン陽に列挙),23.52,24.0,0.0,5231
1,RMP-IP(列生成の13列だけ),24.00,24.0,0.0,1


In [8]:
compact_row = df.loc[df["variant"] == "compact(全パターン陽に列挙)"].iloc[0]
rmp_row = df.loc[df["variant"] == f"RMP-IP(列生成の{gen}列だけ)"].iloc[0]

labels = ["compact\n(全パターン)", f"RMP-IP\n({gen}列)"]
bar_colors = [C["muted"], C["s1"]]
fig = make_subplots(rows=1, cols=3, horizontal_spacing=0.12,
    subplot_titles=("ルート双対境界 (高いほど良い)", "最終 gap [%] (低いほど良い)",
                    "探索ノード数 (少ないほど良い)"))
def add_bars(col, values, fmt):
    fig.add_trace(go.Bar(x=labels, y=values, marker_color=bar_colors,
        text=[fmt(v) for v in values], textposition="outside",
        cliponaxis=False, showlegend=False), row=1, col=col)
add_bars(1, [compact_row["root_dual"], rmp_row["root_dual"]], lambda v: f"{v:.2f}")
add_bars(2, [compact_row["final_gap"]*100, rmp_row["final_gap"]*100], lambda v: f"{v:.1f}%")
add_bars(3, [compact_row["nodes"], rmp_row["nodes"]], lambda v: f"{int(v)}")
fig.update_layout(paper_bgcolor=C["surface"], plot_bgcolor=C["surface"],
    font=dict(family=FONT, color=C["ink2"], size=12), height=360,
    margin=dict(l=40, r=20, t=64, b=40),
    title=dict(text="列生成の before / after(compact vs 生成済み列だけのRMP-IP)",
               x=0.01, font=dict(color=C["ink"], size=15)))
fig.update_yaxes(gridcolor=C["grid"], linecolor=C["axis"], zeroline=False)
fig.update_xaxes(linecolor=C["axis"])
show(fig)

In [9]:
print(f"ルート双対境界 : compact {compact_row['root_dual']:.2f}  vs  RMP-IP {rmp_row['root_dual']:.2f}")
print(f"最終gap        : compact {compact_row['final_gap']:.1%}  vs  RMP-IP {rmp_row['final_gap']:.1%}")
print(f"ノード数       : compact {int(compact_row['nodes'])}  vs  RMP-IP {int(rmp_row['nodes'])}")
print(f"使用パターン数 : compact {total_patterns}(全列挙)  vs  "
      f"RMP-IP {gen}({gen/total_patterns*100:.1f}%、pricingで生成した分だけ)")

ルート双対境界 : compact 23.52  vs  RMP-IP 24.00
最終gap        : compact 0.0%  vs  RMP-IP 0.0%
ノード数       : compact 5231  vs  RMP-IP 1
使用パターン数 : compact 131(全列挙)  vs  RMP-IP 13(9.9%、pricingで生成した分だけ)


## まとめ

- **LP境界の強さそのものは同等**(GG境界と材料下界はほぼ一致)。列生成の価値は
  「LPが強くなること」ではなく、**指数的なパターンを列挙せず、pricingで必要な列だけを
  暗黙に扱えること**にある。この題材では総実行可能パターンのごく一部だけを生成して
  最良のLP境界に到達した(2節末の出力を参照)。
- 生成済みの列だけを使った制限主問題(RMP-IP)は、対称な巨大コンパクト定式化より
  はるかに少ないノードで解ける(4節)。パターンという単位に定式化しなおすこと自体が、
  ロール入れ替えの対称性を最初から消している。

### なぜ SCIP が自動でやらないのか

SCIP は与えられたモデルの列(変数)をそのまま扱う。「パターンを列として動的に生成する」
という再定式化は、コンパクト定式化を書いた時点でモデルの構造(何が変数で何が制約か)が
確定してしまっているため、presolve の範囲では発見できない。列生成はモデラーが
「主問題」と「pricing問題」を設計する分業であり、これは診断で自動検出できる類のものではない
(`decomposable` の recipe として案内はされる)。

### 効かないとき・注意

- **price_and_branch は最適保証がない**。`lp_lb == int_obj` を確認しない限り
  「最適」と言い切らないこと(FINDINGS §4に反例あり: 小規模テストで
  price_and_branch=5 に対し全列挙ILPの真の最適=4)。
- 退化問題(双対 $\pi$ が反復ごとに大きく振動する)では**Wentges安定化**
  (`alpha>0`)が有効。`mk.column_generation(..., alpha=0.5)` のように中心への
  平滑化で反復数を減らせる(過大な `alpha` は過剰安定化で未収束になりうる)。

関連: [手法ガイド 6. 列生成](../../playbook/06-column-generation.md) /
API [`mk.column_generation`/`mk.price_and_branch`](../../api/frameworks.md) /
worked example: `experiments/run_colgen.py` / `run_stabilize.py` / `run_bnp.py`